# Metric views and Genie: companion notebook

This notebook goes with the blog post *Define it once: moving business logic upstream with Databricks metric views* [link to come]. It rebuilds the demo from the post: a small orders table, three analysts' revenue queries, a metric view that holds Finance's definition, and the Genie setups used to compare them.

## Before you start

- You need a Databricks workspace with Unity Catalog, and compute on Databricks Runtime 17.2 or above. The metric view uses YAML version 1.1, which requires it.
- You need permission to create a schema in your catalog, plus `CREATE TABLE` and `USE SCHEMA` on that schema.
- The notebook creates a schema called `metrics_demo` in the `workspace` catalog. To use a different catalog, change the `USE CATALOG` line in the next cell and replace `workspace` in the fully qualified names throughout.
- Run the cells in order, or use **Run all**.

## Set up the schema

In [ ]:
%sql
USE CATALOG workspace;              -- change this if you use a different catalog
 
CREATE SCHEMA IF NOT EXISTS metrics_demo;
USE SCHEMA metrics_demo;
 
SELECT current_catalog(), current_schema();  -- confirm before continuing



## Create and load the orders table

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.metrics_demo.orders (
  order_id STRING,
  order_date DATE,
  customer_id STRING,
  region STRING,
  channel STRING,
  status STRING,
  is_test_order BOOLEAN,
  gross_amount DECIMAL(10,2),
  discount_amount DECIMAL(10,2),
  refund_amount DECIMAL(10,2)
);

INSERT INTO workspace.metrics_demo.orders VALUES
  ('ORD-1001',DATE'2026-01-05','C001','North','Web',     'completed',false, 420.00,20.00,  0.00),
  ('ORD-1002',DATE'2026-01-07','C002','North','Retail',  'completed',false, 180.00, 0.00,  0.00),
  ('ORD-1003',DATE'2026-01-09','C003','South','Web',     'completed',false, 640.00,40.00,  0.00),
  ('ORD-1004',DATE'2026-01-12','C004','South','Partner', 'returned', false, 250.00, 0.00,250.00),
  ('ORD-1005',DATE'2026-01-14','C005','East', 'Web',     'completed',false, 310.00,10.00,  0.00),
  ('ORD-1006',DATE'2026-01-16','C001','East', 'Web',     'completed',false, 150.00, 0.00,  0.00),
  ('ORD-1007',DATE'2026-01-19','C006','West', 'Retail',  'completed',false, 880.00,80.00,  0.00),
  ('ORD-1008',DATE'2026-01-21','C007','West', 'Web',     'cancelled',false,500.00, 0.00,  0.00),
  ('ORD-1009',DATE'2026-01-23','C002','South','Web',     'returned', false, 220.00, 0.00, 20.00),
  ('ORD-1010',DATE'2026-01-26','C008','North','Partner', 'completed',false, 740.00,40.00,  0.00),
  ('ORD-1011',DATE'2026-01-28','C009','East', 'Retail',  'completed',false, 160.00, 0.00,  0.00),
  ('ORD-1012',DATE'2026-01-30','C010','West', 'Web',     'returned', false, 390.00,30.00, 60.00),
  ('ORD-1013',DATE'2026-02-02','C003','North','Web',     'completed',false, 280.00, 0.00,  0.00),
  ('ORD-1014',DATE'2026-02-04','C004','South','Web',     'completed',false, 510.00,10.00,  0.00),
  ('ORD-1015',DATE'2026-02-06','C005','East', 'Partner', 'completed',false, 430.00,30.00,  0.00),
  ('ORD-1016',DATE'2026-02-09','C006','West', 'Web',     'completed',false, 120.00, 0.00,  0.00),
  ('ORD-1017',DATE'2026-02-11','C011','North','Web',     'completed',true, 9999.00,0.00,  0.00),
  ('ORD-1018',DATE'2026-02-13','C007','West', 'Retail',  'completed',false, 660.00,60.00,  0.00),
  ('ORD-1019',DATE'2026-02-16','C008','North','Web',     'returned', false, 340.00, 0.00, 40.00),
  ('ORD-1020',DATE'2026-02-18','C009','East', 'Web',     'cancelled',false,270.00, 0.00,  0.00),
  ('ORD-1021',DATE'2026-02-20','C010','West', 'Partner', 'completed',false, 950.00,50.00,  0.00),
  ('ORD-1022',DATE'2026-02-23','C001','North','Web',     'completed',false, 230.00, 0.00,  0.00),
  ('ORD-1023',DATE'2026-02-25','C011','South','Web',     'completed',true, 5000.00,0.00,  0.00),
  ('ORD-1024',DATE'2026-02-27','C005','East', 'Web',     'returned', false, 370.00,20.00, 70.00);

In [ ]:
%sql
-- Sanity check. This must return 7400.00.
SELECT SUM(gross_amount - discount_amount - refund_amount)
FROM orders
WHERE status <> 'cancelled' AND is_test_order = false;

# The orders table

This table contains 24 orders from January and February 2026, so every figure in this notebook is Q1 to date.

Each row represents one order, with information about:

- the customer
- region and sales channel
- order status
- whether it is a test order
- gross sales
- discounts
- refunds

You've been asked to calculate revenue.

**What would you need to think about before deciding what counts as revenue?**

In [ ]:
%sql
Select * from workspace.metrics_demo.orders;

# Three analysts, three answers

A CFO asks:

> **What's our net revenue for Q1 so far?**

Three analysts query the same orders table:

- **Analyst A:** $23,999
- **Analyst B:** $6,320
- **Analyst C:** $7,400

All three queries run without an error.

**Which answer would you trust, and why?**

**Analyst A** counts the gross amount on every row.

Expected result: $23,999.

In [ ]:
%sql
-- Analyst A
SELECT SUM(gross_amount) AS revenue
FROM workspace.metrics_demo.orders;

**Analyst B** subtracts discounts and counts only completed, non-test orders.

Expected result: $6,320.

In [ ]:
%sql
-- Analyst B: gross sales minus discounts, counting only completed, non-test orders
SELECT
    -- subtract the discount from the gross amount for each order, then sum
    SUM(gross_amount - discount_amount) AS revenue
FROM workspace.metrics_demo.orders  -- the orders table in the metrics_demo schema
WHERE status = 'completed'          -- only keep orders that actually completed
  AND is_test_order = false;        -- exclude test/internal orders

**Analyst C** subtracts discounts and refunds and excludes cancelled and test orders.

Expected result: $7,400.

In [ ]:
%sql
-- Analyst C: net revenue = gross sales minus discounts and refunds,
-- excluding cancelled and test orders (this is the governed definition)
SELECT
    -- subtract discounts and refunds from gross, then sum across all qualifying orders
    SUM(gross_amount - discount_amount - refund_amount) AS revenue
FROM workspace.metrics_demo.orders  -- the orders table in the metrics_demo schema
WHERE status <> 'cancelled'         -- exclude orders that never went through
  AND is_test_order = false;         -- exclude test/internal orders

## The missing business context

**Finance has established the official definition of revenue:**

- Cancelled and test orders do not count.
- Completed orders count as sales.
- Refunds and discounts are deducted from revenue.
- Returned orders still count as sales, but their refunds reduce revenue.

**The correct answer is Analyst C: $7,400.**

```sql
SELECT
    SUM(gross_amount - discount_amount - refund_amount) AS revenue
FROM workspace.metrics_demo.orders
WHERE status <> 'cancelled'
  AND is_test_order = false;
```

## So where does the definition live?

We've agreed on what revenue means. But where do we put that logic so everyone uses it consistently?

The same KPI might appear in:

- a Tableau dashboard
- a scheduled report
- an alert
- an executive summary
- an AI assistant

If each one recreates the revenue logic independently, the definitions can drift.

**A KPI is a business agreement, not a SQL expression. The challenge is making that agreement reusable.**

# Create the metric view

The metric view holds Finance's definition in Unity Catalog: a global filter for cancelled and test orders, three dimensions, and three measures. The last measure is built from the first two. Display names, synonyms, and comments help people and Genie map business terms to the right measure.

You can also build this in Catalog Explorer: open the orders table and choose **Create > Metric view**.

In [ ]:
%sql
CREATE OR REPLACE VIEW workspace.metrics_demo.orders_metric_view
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
source: workspace.metrics_demo.orders
comment: Finance-approved sales metrics excluding cancelled and test orders.
filter: source.status <> 'cancelled' AND NOT source.is_test_order

dimensions:
  - name: customer_id
    expr: source.customer_id
    display_name: Customer ID
  - name: region
    expr: source.region
    display_name: Region
    synonyms:
      - territory
      - sales region
  - name: order_date
    expr: source.order_date
    display_name: Order Date

measures:
  - name: net_revenue
    expr: SUM(source.gross_amount - source.discount_amount - source.refund_amount)
    comment: Finance-approved revenue after discounts and refunds. Cancelled and test orders excluded.
    display_name: Net Revenue
    format:
      type: currency
      currency_code: USD
    synonyms:
      - revenue
      - net sales
  - name: active_customers
    expr: COUNT(DISTINCT source.customer_id)
    comment: Distinct customers with at least one qualifying order in the selected data.
    display_name: Active Customers
    synonyms:
      - customers
      - unique customers
  - name: net_revenue_per_active_customer
    expr: MEASURE(net_revenue) / NULLIF(MEASURE(active_customers), 0)
    comment: Net revenue divided by distinct qualifying customers at the requested grain.
    display_name: Net Revenue per Active Customer
    format:
      type: currency
      currency_code: USD
    synonyms:
      - revenue per customer
      - revenue per active customer
$$

## Query the metric view

Measures go through the `MEASURE()` function. The filter and the formula come from the metric view, so the query doesn't repeat them.

Expected result: $7,400.

In [ ]:
%sql
SELECT
  MEASURE(net_revenue) AS net_revenue
FROM workspace.metrics_demo.orders_metric_view;

### Ask for revenue at a different grain without rewriting the logic

Expected results: East $1,290, North $2,090, South $1,300, and West $2,720.

In [ ]:
%sql
SELECT
  region,
  MEASURE(net_revenue) AS net_revenue
FROM workspace.metrics_demo.orders_metric_view
GROUP BY ALL
ORDER BY region;

### Add the other two measures

Three measures at the regional grain:

- **Net revenue:** sales after discounts and refunds.
- **Active customers:** distinct customers with qualifying orders.
- **Net revenue per active customer:** net revenue divided by the distinct customer count.

The metric view defines each measure. The query only chooses which measures and fields to show.

Expected results:

| Region | Net revenue | Active customers | Revenue per customer |
|---|---:|---:|---:|
| East | $1,290 | 3 | $430.00 |
| North | $2,090 | 4 | $522.50 |
| South | $1,300 | 3 | $433.33 |
| West | $2,720 | 3 | $906.67 |

In [ ]:
%sql
SELECT
  region,
  MEASURE(net_revenue) AS net_revenue,
  MEASURE(active_customers) AS active_customers,
  MEASURE(net_revenue_per_active_customer) AS revenue_per_customer
FROM workspace.metrics_demo.orders_metric_view
GROUP BY ALL
ORDER BY region;

### Ask the same question company-wide

Remove `region` and `GROUP BY` to ask for the measures across the whole company.

Expected results: $7,400 revenue, 10 unique customers, and $740 revenue per customer.

In [ ]:
%sql
SELECT
  MEASURE(net_revenue) AS net_revenue,
  MEASURE(active_customers) AS active_customers,
  MEASURE(net_revenue_per_active_customer) AS revenue_per_customer
FROM workspace.metrics_demo.orders_metric_view;

## Why this matters

Some metrics can't be safely added up across groups.

Here, the regions show **13 active customers**, but there are only **10 unique customers** overall.

A metric view recalculates the measure at the level you ask for, so totals, distinct counts, averages, and ratios stay correct.

**What if we queried the original table instead?**

We can calculate these same regional results in ordinary SQL. But we must remember every part of Finance’s definition: which orders qualify, how refunds and discounts work, how to count unique customers, and how to calculate the ratio.

Expected results: the same as the regional metric view query above.

In [ ]:
%sql
SELECT
    region,
    SUM(gross_amount - discount_amount - refund_amount) AS net_revenue,
    COUNT(DISTINCT customer_id) AS active_customers,
    SUM(gross_amount - discount_amount - refund_amount)
        / NULLIF(COUNT(DISTINCT customer_id), 0) AS revenue_per_customer
FROM workspace.metrics_demo.orders
WHERE status <> 'cancelled'
    AND is_test_order = false
GROUP BY region
ORDER BY region;

# Test Genie with and without the metric view

Build the same Genie Agent three times. For each one, start a new chat and ask the same question: **Tell me about total revenue.** Open **Show code** on each answer to see the SQL Genie generated.

## 1. The orders table, no instructions

Create a Genie Agent with `workspace.metrics_demo.orders` as its only data source and no general instructions.

Genie has to decide what revenue means on its own, so this answer can vary between runs. In the run for the blog post, it returned $8,170: it excluded test orders but kept cancelled ones.

## 2. The orders table, with instructions

Keep `workspace.metrics_demo.orders` as the only data source, and paste this into the agent's general instructions:

```text
When calculating net revenue:
- Exclude test orders where is_test_order = true.
- Exclude cancelled orders.
- Returned orders still count as sales.
- Calculate net revenue as gross_amount - discount_amount - refund_amount.
```

Expected result: $7,400.

## 3. The metric view, no instructions

Remove the orders table and the instructions, and add `workspace.metrics_demo.orders_metric_view` as the only data source.

Expected result: $7,400, with generated SQL that calls `MEASURE(net_revenue)`.

## Results from the blog post

| Data source | Additional instructions | Observed revenue |
|---|---|---:|
| Orders table | None | $8,170 |
| Orders table | Finance's definition | $7,400 |
| Metric view | None | $7,400 |

These results come from one workspace, not a guarantee. A metric view reduces the decisions Genie has to make about how revenue is calculated, but Genie can still misread a question or choose the wrong measure, so check the generated SQL.